In [1]:
import sys
from ortools.sat.python import cp_model

In [2]:
def read_input():
    input = list(map(int, sys.stdin.read().split()))
    idx = 0
    m = input[idx]; idx += 1
    n = input[idx]; idx += 1

    preference = [[0] * n for _ in range(m)]
    for i in range(m):
        e = input[idx]; idx += 1
        for _ in range(e):
            c = input[idx] - 1; idx += 1
            preference[i][c] = 1

    conflict = []
    k = input[idx]; idx += 1
    for _ in range(k):
        a = input[idx] - 1; idx += 1
        b = input[idx] - 1; idx += 1
        conflict.append((a, b))

    return m, n, preference, conflict

In [ ]:
def solve(m, n, preference, conflict):
    model = cp_model.CpModel()
    solver = cp_model.CpSolver()

    x = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            x[i][j] = model.new_bool_var(f"x[{i}][{j}]")

    for j in range(n):
        model.add(sum(x[i][j] for i in range(m)) == 1)

    for i in range(m):
        for j in range(n):
            if not preference[i][j]:
                 model.add(x[i][j] == 0)

    for (i, j) in conflict:
        for t in range(m):
            model.add(x[t][i] + x[t][j] <= 1)

    load = [0] * m
    for t in range(m):
        load[t] = sum(x[t][i] for i in range(n))
    
    max_load = model.new_int_var(0, n, f"max_load")
    for t in range(m):
        model.add(max_load >= load[t])
    model.minimize(max_load)

    solver.parameters.max_time_in_seconds = 5.0
    status = solver.solve(model)
    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        print(solver.objective_value)
        for i in range(m):
            print(f"Courses of teacher {i}:")
            for j in range(n):
                
                if solver.value(x[i][j]) == 1:
                    print(j, end=" ")
            print()
    else:
        print(-1)
        

In [8]:
if __name__ == "__main__":
    f = open("input.txt", "r")
    sys.stdin = f
    m, n, preference, conflict = read_input()
    solve(m, n, preference, conflict)
    f.close()

5.0
Courses of teacher 0:
3 4 8 10 
Courses of teacher 1:
0 1 5 6 
Courses of teacher 2:
2 7 9 11 12 
